<a href="https://colab.research.google.com/github/MichalSlowakiewicz/NLP/blob/master/nlp_hw1_speculative_decoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a target="_blank" href="https://colab.research.google.com/github/mim-ml-teaching/public-nlp-2025-26/blob/main/hws/NLP_HW1_Speculative_Decoding.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab">
</a>

# NLP Homework 1: Speculative Decoding

## Background

In standard autoregressive transformers, tokens are generated strictly one at a time. Each new token requires a full forward pass, and because generation is causal, the model cannot produce the $k^\text{th}$ token until all preceding tokens have been processed.

This bottleneck is addressed by *Speculative decoding*. A smaller, faster "draft" model first proposes a sequence of candidate tokens. The larger target model then evaluates it in a single forward pass. If the proposed tokens are in fact assigned high-probabilities by a target model then they are accepted, saving additional computation. If some of the proposed tokens are incorrect, the sequence of tokens is rejected at the point of divergence. Although this requires partial recomputation, the larger model has already processed the prefix in parallel.

**Core idea** Given a prefix, a cheap *draft model* proposes $K$ subsequent tokens: $t_1, \dots, t_K$.
The expensive *target model* is then run **once** on the extended sequence: $[\text{prefix}, t_1, \dots, t_K]$.
This single forward pass produces logits for all $K+1$ positions in parallel.

**Verification**  The target model ($p_{\text{target}}(\cdot \mid \text{prefix})$) is used to verify the proposed tokens. That is,
for $i = 1, \dots, K$, we check whether
$$
t_i = \arg\max\; p_{\text{target}}(\cdot \mid \text{prefix}, t_1, \dots, t_{i-1}).
$$
This can be done in parallel! Now, let $j$ be the largest index such that for all $t_1, \dots, t_j$ the above equality holds. It turns out, we can accept $t_1, \dots, t_j$, discard the rest, and also take the target model's prediction at position $j+1$.

## Tasks

You will work with WikiText-2 (`wikitext-2-raw-v1`) dataset, and target (verifier) model `google/gemma-3-4b-pt`.


### Task 1: Implement Speculative Decoding Loop (4 pts)

Implement two functions:

1. **`verify_draft_tensor(context_ids, draft_ids, model)`** - Given a context and a sequence of $K$ draft tokens, run a single forward pass of the target model on the concatenated sequence and determine how many draft tokens match the target model's greedy predictions. Return the accepted prefix plus the target model's next-token prediction. **Your implementation must be fully vectorized (no `for` loops)** to receive maximum number of points.

2. **`speculative_generate_tensor(context_ids, draft_fn, model, n_tokens, k)`** - The outer generation loop that repeatedly calls the draft function and verifier until `n_tokens` new tokens have been generated. Track and return statistics (e.g., total draft tokens proposed, total accepted, number of verification rounds). Try to avoid device to host transfers in the main loop.

See the code cell below for the exact function signatures.

### Task 2: Design a Draft Model & Evaluate (4 pts)

Design and implement a **transformer-based** draft model that proposes $K$ candidate tokens given a context. You may either:

- Use a pre-trained small model (e.g., `google/gemma-3-1b-pt`), or
- Train a small transformer from scratch (e.g., via distillation from the target model, or training directly on `wikitext-2-raw-v1`).

**Deliverables:**

1. Implement your draft model and wrap it so it conforms to the `draft_fn(context_ids, k)` interface (see `speculative_generate_tensor` argument, or `bigram_draft` function).
2. Evaluate on the WikiText-2 **test** split (`test_ds`).
3. Report the acceptance rate (fraction of draft tokens accepted by the target model).
4. Compare against the provided bigram baseline - your model should achieve a higher acceptance rate.
5. Measure wall time of speculative decoding vs standard execution. Your solution should achieve a speedup.

### Task 3: Visualization & Report (2 pts)

Write a concise summary (≤ 300 words) describing your approach, key findings, and observations.

Your summary should include at least two informative plots. For example you may visualize:

* Speedup vs model size (if you train your own transformer)
* Acceptance rate vs context length for draft model
* Histogram of accepted draft lengths

All plots must have axis labels, legends, and titles.

In [1]:
# added to integrate with Hugging Face + additional imports
from huggingface_hub import login
from google.colab import userdata
import time
import matplotlib.pyplot as plt

login(token=userdata.get('token'))

In [2]:
import random
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from collections import defaultdict

random.seed(42)
torch.manual_seed(42)

TARGET_MODEL = "google/gemma-3-4b-pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.bfloat16 if device == "cuda" else torch.float32
print(f"Running on: {device}")

def sync():
    if device == "cuda":
        torch.cuda.synchronize()

Running on: cuda


In [3]:
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)
target_model = AutoModelForCausalLM.from_pretrained(TARGET_MODEL, torch_dtype=dtype).to(device)
target_model.eval()

n_params_target = sum(p.numel() for p in target_model.parameters()) / 1e6
print(f"Target model: {TARGET_MODEL}  ({n_params_target:.0f}M params)  on {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/815 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Target model: google/gemma-3-4b-pt  (4300M params)  on cuda


In [4]:
@torch.no_grad() # added to save VRAM
def autoregressive_generate(context_ids: list[int], n_tokens: int) -> list[int]:
    """Standard autoregressive generation with the target model (baseline)."""
    input_ids = torch.tensor([context_ids], device=device)
    for _ in range(n_tokens):
        logits = target_model(input_ids).logits
        next_tok = logits[:, -1:].argmax(dim=-1)
        input_ids = torch.cat([input_ids, next_tok], dim=1)
    return input_ids[0, len(context_ids):].tolist()

In [5]:
def get_split(split):
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split=split)
    examples = [row["text"] for row in ds if row["text"].strip()]
    tokenized_examples = [tokenizer.encode(text, add_special_tokens=False) for text in examples]
    num_tokens = sum(len(tokens) for tokens in tokenized_examples)
    print(f"{split.capitalize()} split: {len(examples):,} examples, {num_tokens:,} tokens in total.")
    return tokenized_examples

train_ds = get_split("train")
test_ds = get_split("test")

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Train split: 23,767 examples, 2,459,301 tokens in total.
Test split: 2,891 examples, 292,281 tokens in total.


# Naive draft model (bigrams trained on the dataset)


In [6]:
train_ids = [tok for example in train_ds for tok in example]
bigram_counts = defaultdict(lambda: defaultdict(int))
for id1, id2 in zip(train_ids, train_ids[1:]):
    bigram_counts[id1][id2] += 1

bigram_next = {}
for tok, successors in bigram_counts.items():
    bigram_next[tok] = max(successors, key=successors.get)

fallback_token = tokenizer.encode("the", add_special_tokens=False)[0]

def bigram_draft(context_ids: list[int], k: int) -> list[int]:
    draft = []
    last_tok = context_ids[-1]
    for _ in range(k):
        next_tok = bigram_next.get(last_tok, fallback_token)
        draft.append(next_tok)
        last_tok = next_tok
    return draft

In [7]:
# Example run

prompt = "The history of artificial"
ctx = tokenizer.encode(prompt, add_special_tokens=False)

draft = bigram_draft(ctx, k=3)
print(f"Context: {prompt!r}")
print(f"Bigram draft: {tokenizer.decode(draft)!r}")

Context: 'The history of artificial'
Bigram draft: ' intelligence . '


# Task 1: Implement speculative decoding

In [8]:
@torch.no_grad()
def verify_draft_tensor(
    context_ids: torch.Tensor,
    draft_ids: torch.Tensor,
    model,
) -> torch.Tensor:
    """
    Verify draft tokens against a model.

    Args:
        context_ids: 1-D tensor (n,) — context tokens.
        draft_ids:   1-D tensor (k,) — drafted token ids to verify.
        model:       the verifier model.

    Returns:
        1-D tensor — verified context + accepted predictions + next token.
    """
    # TODO
    N, K = context_ids.shape[0], draft_ids.shape[0]
    combined = torch.cat([context_ids, draft_ids])
    combined = combined.view(1, N+K)

    # obtaining predictions of the model for new tokens
    logits = model(combined).logits
    predictions = torch.argmax(logits, dim=2).view(N+K)
    predictions_reduced = predictions[-K-1:-1]

    # checking which tokens from draft prediction align with model's prediction
    agreement_vector = torch.zeros_like(draft_ids)
    mask = (predictions_reduced == draft_ids)
    agreement_vector[mask] = 1
    first_zero_index = (agreement_vector == 0).int().argmax().item()

    # returning new tensor up to the point where predictions didn't agree
    if agreement_vector.sum() == K:
      return torch.cat([context_ids, predictions[-K-1:]])
    else:
      return torch.cat([context_ids, predictions[-K-1:-K+first_zero_index]])

@torch.no_grad()
def speculative_generate_tensor(
    context_ids: list[int],
    draft_fn,
    model,
    n_tokens: int,
    k: int,
) -> tuple[list[int], dict]:
    """
    Speculative generation.

    Args:
        context_ids: prefix token ids.
        draft_fn:    callable(sequence_tensor, k) -> (k,) device tensor.
        model:       verifier model.
        n_tokens:    number of tokens to generate.
        k:           speculation length.

    Returns:
        (generated_token_ids, stats_dict)
    """
    # TODO
    total_drafted = 0
    total_accepted = 0
    n_rounds = 0
    current_sequence = torch.tensor(context_ids, device=device, dtype=torch.long)
    desired_length = len(context_ids) + n_tokens

    # loop for sequential generation using the draft model
    while len(current_sequence) < desired_length:
      new_tokens_draft = draft_fn(current_sequence, k)
      verified_tensor = verify_draft_tensor(current_sequence, new_tokens_draft, model)
      total_drafted += k
      total_accepted += len(verified_tensor) - 1 - len(current_sequence)
      n_rounds += 1
      current_sequence = verified_tensor

    # extracting only new tokens
    results_tensor = verified_tensor[len(context_ids):]

    # protection if we generated more than n new tokens
    if len(results_tensor) > n_tokens:
      results_tensor = results_tensor[:n_tokens]

    # preparing dictionary with statistics
    stats_dict = {'Tokens drafted': total_drafted, 'Tokens accepted': total_accepted, 'Number of rounds': n_rounds}
    results_list = results_tensor.tolist()

    return results_list, stats_dict

# Task 2: Evaluate speculative decoding


In [9]:
# TODO

In [10]:
# importing smaller draft model
DRAFT_MODEL = "google/gemma-3-1b-pt"
draft_model = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL, torch_dtype=dtype).to(device)
draft_model.eval()

config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

In [11]:
def draft_function(sequence_tensor, k):
  sequence_tensor = sequence_tensor.view(1, -1)

  with torch.no_grad():
    # 'num_beams=1' and 'do_sample=False' were added to set greedy prediction type
    model_output = draft_model.generate(inputs=sequence_tensor, max_new_tokens=k, num_beams=1, do_sample=False) # this returns original context + new tokens

  # reducing output only to new tokens
  new_tokens = model_output[:, -k:]
  new_tokens = new_tokens.view(-1)

  return new_tokens

In [12]:
# evaluation function
def evaluation_function(n_new_tokens, test_ds, draft_function, k):
  """
  Evaluation function, which returns 6 statistics:
  (acceptance_rate, avg_time_standard, avg_time_speculative_decoding, avg_time_diff, context_lengths, sequence_acceptance_rates)
  """
  # initializing variables and lists
  total_drafted_all = 0
  total_accepted_all = 0
  n_rounds_all = 0
  time_diffs = []
  times_standard = []
  times_speculative_decoding = []
  context_lengths = []
  sequence_acceptance_rates = []

  # main loop across the test dataset
  for context in test_ds:

    # speculative decoding with the use of draft model
    sync()
    t1_start = time.time()
    generated_tokens_speculative_decoding, stats_dict = speculative_generate_tensor(context_ids=context, draft_fn=draft_function, model=target_model, n_tokens=n_new_tokens, k=k)
    sync()
    t1_end = time.time()
    t1 = t1_end - t1_start

    # standard autoregressive method
    sync()
    t2_start = time.time()
    generated_tokens_standard = autoregressive_generate(context, n_new_tokens)
    sync()
    t2_end = time.time()
    t2 = t2_end - t2_start

    # updating lists and statistics
    time_diffs.append(t1-t2)
    times_standard.append(t2)
    times_speculative_decoding.append(t1)

    drafted = stats_dict['Tokens drafted']
    accepted = stats_dict['Tokens accepted']

    context_lengths.append(len(context))
    sequence_acceptance_rates.append(accepted / drafted if drafted > 0 else 0)

    total_drafted_all += drafted
    total_accepted_all += accepted
    n_rounds_all += stats_dict['Number of rounds']

  # calculating final statistics
  acceptance_rate = total_accepted_all / total_drafted_all
  avg_time_standard = sum(times_standard) / len(times_standard)
  avg_time_speculative_decoding = sum(times_speculative_decoding) / len(times_speculative_decoding)
  avg_time_diff = sum(time_diffs) / len(time_diffs)

  print("Acceptance rate:", acceptance_rate)
  print("Average time (standard approach):", avg_time_standard)
  print("Average time (speculative decoding approach):", avg_time_speculative_decoding)
  print("Average time difference (speculative decoding - standard time):", avg_time_diff)

  return acceptance_rate, avg_time_standard, avg_time_speculative_decoding, avg_time_diff, context_lengths, sequence_acceptance_rates

In [15]:
acceptance_rate_gemma, avg_time_standard_gemma, avg_time_speculative_decoding_gemma, avg_time_diff_gemma, context_lengths_gemma, sequence_acceptance_rates_gemma  = evaluation_function(n_new_tokens=30, test_ds=test_ds[:100], draft_function=draft_function, k=5)

Acceptance rate: 0.928996282527881
Average time (standard approach): 15.806707816123962
Average time (speculative decoding approach): 4.951637349128723
Average time difference (speculative decoding - standard time): -10.855070466995238


In [16]:
def bigram_draft_adjusted(sequence_tensor, k):
  """
  Wrapper function for Bigram draft model.
  Ensures compatibility with 'evaluation_function'.
  """
  sequence_list = sequence_tensor.tolist()
  draft = bigram_draft(sequence_list, k)
  draft_tensor = torch.tensor(draft, dtype=torch.long, device=device)
  return draft_tensor

In [ ]:
acceptance_rate_bigram, avg_time_standard_bigram, avg_time_speculative_decoding_bigram, avg_time_diff_bigram, context_lengths_bigram, sequence_acceptance_rates_bigram = evaluation_function(n_new_tokens=30, test_ds=test_ds[:100], draft_function=bigram_draft_adjusted, k=5)

In [ ]:
len(test_ds)


# Task 3: Visualization & Report


In [ ]:
# TODO: Your plots

In [ ]:
# Plot 1: Acceptance Rate comparison
draft_models = ["Bigram (Baseline)", "Gemma 1B"]
results = [acceptance_rate_bigram, acceptance_rate_gemma]

plt.figure(figsize=(12,6))
plt.bar(draft_models, results, color=['red', 'blue'])
plt.title("Acceptance Rate comparison: Bigram vs Gemma")
plt.ylabel("Acceptance Rate")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot 2: Average generation time
methods = ['Standard AR', 'SD Bigram', 'SD Gemma']
results = [(avg_time_standard_gemma + avg_time_standard_bigram) / 2, avg_time_speculative_decoding_bigram, avg_time_speculative_decoding_gemma]

plt.figure(figsize=(12,6))
plt.bar(methods, results, color=['green', 'red', 'blue'])
plt.title("Average generation time for 30 new tokens")
plt.ylabel("Time (seconds)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot 3.1: Acceptance rate vs context length (Gemma)
plt.figure(figsize=(12,6))
plt.scatter(context_lengths_gemma, sequence_acceptance_rates_gemma, alpha=0.7, color='blue')
plt.title("Acceptance rate vs context length for draft model Gemma")
plt.xlabel("Context length")
plt.ylabel("Acceptance Rate")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot 3.2: Acceptance rate vs context length (Bigram)
plt.figure(figsize=(12,6))
plt.scatter(context_lengths_bigram, sequence_acceptance_rates_bigram, alpha=0.7, color='red')
plt.title("Acceptance rate vs context length for draft model Bigram (Baseline)")
plt.xlabel("Context length")
plt.ylabel("Acceptance Rate")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# short experiment to examine the impact of 'k' parameter on Acceptance Rate and Average Time
k_values = [2, 4, 6, 8, 10]
results_acceptance_rate = []
results_average_time = []

for k in k_values:
  acceptance_rate_gemma_experiment, _, avg_time_speculative_decoding_experiment, _, _, _ = evaluation_function(n_new_tokens=30, test_ds=test_ds[:20], draft_function=draft_function, k=k)
  results_acceptance_rate.append(acceptance_rate_gemma_experiment)
  results_average_time.append(avg_time_speculative_decoding_experiment)

# Plot 4: Impact of K - Acceptance Rate
plt.figure(figsize=(12,6))
plt.plot(k_values, results_acceptance_rate, marker='o', linestyle='-', color='orange')
plt.title("Impact of parameter K on Acceptance Rate")
plt.xlabel("Parameter K")
plt.ylabel("Average Acceptance Rate")
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.show()

# Plot 5: Impact of K - Average Time
plt.figure(figsize=(12,6))
plt.plot(k_values, results_average_time, marker='o', linestyle='-', color='blue')
plt.title("Impact of parameter K on Average Time")
plt.xlabel("Parameter K")
plt.ylabel("Average Time (seconds)")
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.show()

*TODO: Write your report here.*